In [ ]:
# ============================================================
# NB_TRNSF_SilverADVWDB_SalesOrderDetail
# Capa: Silver | Origen: ADVWDB | Tabla: SalesOrderDetail
# Carga INCREMENTAL con UPSERT (MERGE Delta Lake)
# FLUJO: CopyData (SQL→lh_silver staging) → este NB → lh_silver MERGE
# FIX: el CopyData deposita en lh_silver.<origen>.<tabla>_staging
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
_ctx    = notebookutils.runtime.context
WS_ID   = _ctx['workspaceId']
_lh_map = {lh.displayName: lh.id for lh in notebookutils.lakehouse.list()}
ART_SILVER = _lh_map.get('lh_silver', '')


# ── IDs técnicos (única excepción documentada en TP) ─────────

# ── Config desde lh_config ───────────────────────────────────
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen       = 'ADVWDB'
      AND nombre_tabla = 'SalesOrderDetail'
      AND activo       = 1
    LIMIT 1
""").collect()[0]

nombre_tabla  = config['nombre_tabla']
origen        = config['origen']
pks           = [pk.strip() for pk in config['pks'].split(',')]
params_inc    = config['parametros_incrementales']
tipo_carga    = config['tipo_carga']
ultimo_valor  = config['ultimo_valor_incremental'] or '1900-01-01 00:00:00'

# Nombres de tabla construidos dinámicamente
tabla_staging = f'lh_silver.{origen}.{nombre_tabla}_staging'
tabla_silver  = f'lh_silver.{origen}.{nombre_tabla}'

# Path ABFS para DeltaTable.forPath (MERGE requiere path físico)
abfs_silver = (
    f'abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com'
    f'/{ART_SILVER}/Tables/{origen}/{nombre_tabla}'
)

print('✅ Config cargada')
print(f'   nombre_tabla  : {nombre_tabla}')
print(f'   origen        : {origen}')
print(f'   pks           : {pks}')
print(f'   params_inc    : {params_inc}')
print(f'   ultimo_valor  : {ultimo_valor}')
print(f'   tabla_staging : {tabla_staging}')
print(f'   tabla_silver  : {tabla_silver}')


In [ ]:
# ============================================================
# BONUS: Lógica de Reproceso
# ============================================================
try:
    fecha_inicio_reproceso = notebookutils.notebook.getParameter('fecha_inicio_reproceso', '')
    fecha_fin_reproceso    = notebookutils.notebook.getParameter('fecha_fin_reproceso', '')
except Exception:
    fecha_inicio_reproceso = ''
    fecha_fin_reproceso    = ''

modo_reproceso = bool(fecha_inicio_reproceso and fecha_fin_reproceso)
if modo_reproceso:
    print(f"REPROCESO Silver SOD: {fecha_inicio_reproceso} → {fecha_fin_reproceso}")
    spark.sql(
        "DELETE FROM lh_silver." + origen + "." + nombre_tabla +
        " WHERE " + params_inc + " >= '" + fecha_inicio_reproceso + "'" +
        " AND "   + params_inc + " <= '" + fecha_fin_reproceso    + "'"
    )
    spark.sql(
        "UPDATE lh_config.dbo.origin_bronze_silver" +
        " SET ultimo_valor_incremental = '" + fecha_inicio_reproceso + "'" +
        " WHERE nombre_tabla = '" + nombre_tabla + "' AND origen = '" + origen + "'"
    )
    ultimo_valor = fecha_inicio_reproceso
    print("   Silver y watermark reseteados para reproceso")
else:
    print("Modo incremental normal")


In [ ]:
# ============================================================
# Celda 2 — Leer desde STAGING via metastore
# FIX: spark.sql con nombre de tabla en lugar de .load(abfs)
# Garantiza que se lee exactamente lo que Copy Data escribió
# LIMIT 1000 obligatorio durante desarrollo (TP)
# ============================================================

try:
    df_raw = spark.sql(f'SELECT * FROM {tabla_staging} LIMIT 1000')
    filas_staging = df_raw.count()
    print(f'✅ Filas desde staging ({tabla_staging}): {filas_staging}')
    display(df_raw.limit(5))
except Exception as e:
    raise Exception(
        f'ERROR: No se encontró {tabla_staging}.\n'
        f'Verificar que DP_INGST_SilverADVWDB haya corrido y el staging esté en lh_silver.\n'
        f'Error original: {e}'
    )

if filas_staging == 0:
    print('ℹ️  Sin registros nuevos en staging — sin datos desde ultimo_valor')
    notebookutils.notebook.exit('sin_datos_nuevos')


In [ ]:
# ============================================================
# Celda 3 — Tipado correcto y limpieza
# Castear cada columna al tipo correcto según dominio ADVW
# PKs y columnas de limpieza vienen de config
# ============================================================

df_silver = (
    df_raw
    .withColumn('SalesOrderID',         F.col('SalesOrderID').cast('integer'))
    .withColumn('SalesOrderDetailID',   F.col('SalesOrderDetailID').cast('integer'))
    .withColumn('CarrierTrackingNumber',F.trim(F.col('CarrierTrackingNumber')))
    .withColumn('OrderQty',             F.col('OrderQty').cast('smallint'))
    .withColumn('ProductID',            F.col('ProductID').cast('integer'))
    .withColumn('SpecialOfferID',       F.col('SpecialOfferID').cast('integer'))
    .withColumn('UnitPrice',            F.col('UnitPrice').cast('decimal(18,4)'))
    .withColumn('UnitPriceDiscount',    F.col('UnitPriceDiscount').cast('decimal(18,4)'))
    .withColumn('LineTotal',            F.col('LineTotal').cast('decimal(38,6)'))
    .withColumn('ModifiedDate',         F.col('ModifiedDate').cast('timestamp'))
)

# PKs desde config — nunca hardcodeadas
df_silver = df_silver.dropna(subset=pks).dropDuplicates(pks)

filas_limpias = df_silver.count()
print(f'✅ Filas después de limpieza y tipado: {filas_limpias}')
display(df_silver.select(
    'SalesOrderID','SalesOrderDetailID',
    'ProductID','OrderQty','UnitPrice','LineTotal','ModifiedDate'
).limit(5))


In [ ]:
# ============================================================
# Celda 4 — UPSERT MERGE Delta Lake
# Condición de merge construida dinámicamente desde PKs en config
# Primera carga: overwrite (tabla no existe aún)
# Cargas siguientes: MERGE (update si existe, insert si es nuevo)
# ============================================================

# Condición dinámica desde PKs leídas de config
merge_condition = ' AND '.join(
    [f'target.{pk} = source.{pk}' for pk in pks]
)
print(f'🔀 Merge condition: {merge_condition}')

try:
    target = DeltaTable.forPath(spark, abfs_silver)
    (
        target.alias('target')
        .merge(df_silver.alias('source'), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print('✅ MERGE ejecutado correctamente')

except Exception:
    # Primera carga — tabla Silver no existe todavía
    print('ℹ️  Primera carga — creando tabla Silver con overwrite inicial')
    (
        df_silver.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(tabla_silver)
    )
    print('✅ Primera carga completada')


In [ ]:
# ============================================================
# Celda 5 — Actualizar watermark en config
# MAX(ModifiedDate) desde Silver post-MERGE
# Actualiza ultimo_valor_incremental para próxima ejecución
# ============================================================

max_fecha = spark.sql(
    f'SELECT MAX({params_inc}) as max_val FROM {tabla_silver}'
).collect()[0]['max_val']

if max_fecha:
    max_fecha_str = str(max_fecha)
    spark.sql(f"""
        UPDATE lh_config.dbo.origin_bronze_silver
        SET ultimo_valor_incremental = '{max_fecha_str}'
        WHERE origen       = '{origen}'
          AND nombre_tabla = '{nombre_tabla}'
    """)
    print(f'✅ Watermark actualizado: {ultimo_valor} → {max_fecha_str}')
else:
    print('⚠️  Sin datos en Silver — watermark sin cambios')

# Limpiar staging (DROP TABLE limpia metastore + archivos)
spark.sql(f'DROP TABLE IF EXISTS {tabla_staging}')
print(f'🗑️  Staging eliminado: {tabla_staging}')


In [ ]:
# ============================================================
# Celda 6 — Verificar resultado final
# ============================================================

total = spark.sql(
    f'SELECT COUNT(*) as cnt FROM {tabla_silver}'
).collect()[0]['cnt']

print(f'✅ Total filas en Silver ({tabla_silver}): {total}')

display(spark.sql(f"""
    SELECT
        CAST(ModifiedDate AS DATE) AS fecha_modificacion,
        COUNT(*)                   AS cant_filas,
        SUM(LineTotal)             AS total_ventas,
        MIN(SalesOrderID)          AS min_orden,
        MAX(SalesOrderID)          AS max_orden
    FROM {tabla_silver}
    GROUP BY CAST(ModifiedDate AS DATE)
    ORDER BY fecha_modificacion
    LIMIT 1000
"""))
